In [4]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.datasets


class NeuralNetwork:
    """
    A modular 3-layer neural network implementing standard Gradient Descent,
    Momentum, and Adam optimizers from scratch using pure NumPy.
    """
    def __init__(self, layer_dims):
        self.layer_dims = layer_dims
        self.L = len(layer_dims) - 1
        self.parameters = self._initialize_parameters()
        
    def _initialize_parameters(self):
        np.random.seed(3)
        parameters = {}
        for l in range(1, self.L + 1):
            # He initialization for ReLU activation layers
            parameters[f"W{l}"] = np.random.randn(self.layer_dims[l], self.layer_dims[l-1]) * np.sqrt(2 / self.layer_dims[l-1])
            parameters[f"b{l}"] = np.zeros((self.layer_dims[l], 1))
        return parameters

    @staticmethod
    def _sigmoid(Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    @staticmethod
    def _relu(Z):
        return np.maximum(0, Z)

    def forward(self, X):
        caches = []
        A = X
        for l in range(1, self.L):
            A_prev = A
            Z = np.dot(self.parameters[f"W{l}"], A_prev) + self.parameters[f"b{l}"]
            A = self._relu(Z)
            caches.append((A_prev, Z))
            
        # Output layer with Sigmoid
        Z = np.dot(self.parameters[f"W{self.L}"], A) + self.parameters[f"b{self.L}"]
        AL = self._sigmoid(Z)
        caches.append((A, Z))
        return AL, caches

    def compute_cost(self, AL, Y):
        m = Y.shape[1]
        cost = -1/m * np.sum(Y * np.log(AL + 1e-15) + (1 - Y) * np.log(1 - AL + 1e-15))
        return np.squeeze(cost)

    def backward(self, AL, Y, caches):
        grads = {}
        m = Y.shape[1]
        dZ = AL - Y
        
        # Output Layer Gradients
        A_prev, Z = caches[-1]
        grads[f"dW{self.L}"] = 1/m * np.dot(dZ, A_prev.T)
        grads[f"db{self.L}"] = 1/m * np.sum(dZ, axis=1, keepdims=True)
        
        # Hidden Layer Backpropagation
        dA = np.dot(self.parameters[f"W{self.L}"].T, dZ)
        for l in reversed(range(1, self.L)):
            A_prev, Z = caches[l-1]
            dZ = np.multiply(dA, np.int64(Z > 0))
            grads[f"dW{l}"] = 1/m * np.dot(dZ, A_prev.T)
            grads[f"db{l}"] = 1/m * np.sum(dZ, axis=1, keepdims=True)
            if l > 1:
                dA = np.dot(self.parameters[f"W{l}"].T, dZ)
                
        return grads

    def _init_optimizer_states(self, optimizer):
        v, s = {}, {}
        for l in range(1, self.L + 1):
            v[f"dW{l}"] = np.zeros_like(self.parameters[f"W{l}"])
            v[f"db{l}"] = np.zeros_like(self.parameters[f"b{l}"])
            s[f"dW{l}"] = np.zeros_like(self.parameters[f"W{l}"])
            s[f"db{l}"] = np.zeros_like(self.parameters[f"b{l}"])
        return v, s

    def train(self, X, Y, optimizer="adam", learning_rate=0.0007, mini_batch_size=64, 
              epochs=10000, beta=0.9, beta1=0.9, beta2=0.999, epsilon=1e-8, print_cost=True):
        
        v, s = self._init_optimizer_states(optimizer)
        costs = []
        t = 0
        seed = 0

        for epoch in range(epochs):
            seed += 1
            np.random.seed(seed)
            m = X.shape[1]
            
            # Mini-Batch Shuffle & Partition
            permutation = list(np.random.permutation(m))
            shuffled_X = X[:, permutation]
            shuffled_Y = Y[:, permutation].reshape((1, m))
            
            num_batches = math.floor(m / mini_batch_size)
            for k in range(0, num_batches + 1):
                start_idx = k * mini_batch_size
                end_idx = min((k + 1) * mini_batch_size, m)
                if start_idx >= end_idx:
                    continue
                    
                mb_X = shuffled_X[:, start_idx:end_idx]
                mb_Y = shuffled_Y[:, start_idx:end_idx]

                # Step 1: Forward Pass
                AL, caches = self.forward(mb_X)
                
                # Step 2: Compute Cost
                cost = self.compute_cost(AL, mb_Y)

                # Step 3: Backward Pass
                grads = self.backward(AL, mb_Y, caches)

                # Step 4: Parameter Updates
                t += 1
                for l in range(1, self.L + 1):
                    if optimizer == "gd":
                        self.parameters[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
                        self.parameters[f"b{l}"] -= learning_rate * grads[f"db{l}"]

                    elif optimizer == "momentum":
                        v[f"dW{l}"] = beta * v[f"dW{l}"] + (1 - beta) * grads[f"dW{l}"]
                        v[f"db{l}"] = beta * v[f"db{l}"] + (1 - beta) * grads[f"db{l}"]
                        self.parameters[f"W{l}"] -= learning_rate * v[f"dW{l}"]
                        self.parameters[f"b{l}"] -= learning_rate * v[f"db{l}"]

                    elif optimizer == "adam":
                        # First moment
                        v[f"dW{l}"] = beta1 * v[f"dW{l}"] + (1 - beta1) * grads[f"dW{l}"]
                        v[f"db{l}"] = beta1 * v[f"db{l}"] + (1 - beta1) * grads[f"db{l}"]
                        v_corrected_W = v[f"dW{l}"] / (1 - beta1**t)
                        v_corrected_b = v[f"db{l}"] / (1 - beta1**t)

                        # Second moment
                        s[f"dW{l}"] = beta2 * s[f"dW{l}"] + (1 - beta2) * (grads[f"dW{l}"]**2)
                        s[f"db{l}"] = beta2 * s[f"db{l}"] + (1 - beta2) * (grads[f"db{l}"]**2)
                        s_corrected_W = s[f"dW{l}"] / (1 - beta2**t)
                        s_corrected_b = s[f"db{l}"] / (1 - beta2**t)

                        # Update
                        self.parameters[f"W{l}"] -= learning_rate * (v_corrected_W / (np.sqrt(s_corrected_W) + epsilon))
                        self.parameters[f"b{l}"] -= learning_rate * (v_corrected_b / (np.sqrt(s_corrected_b) + epsilon))

            if print_cost and epoch % 1000 == 0:
                print(f"Cost after epoch {epoch}: {cost:.6f}")
                costs.append(cost)

        return costs

    def predict(self, X):
        AL, _ = self.forward(X)
        return (AL > 0.5).astype(int)


# --- Execution Example ---
if __name__ == "__main__":
    import math
    
    # 1. Generate Synthetic Dataset
    np.random.seed(3)
    train_X, train_Y = sklearn.datasets.make_moons(n_samples=300, noise=.2)
    train_X = train_X.T
    train_Y = train_Y.reshape((1, train_Y.shape[0]))

    # 2. Initialize Model Architecture [Input, Hidden_1, Hidden_2, Output]
    nn = NeuralNetwork(layer_dims=[train_X.shape[0], 5, 2, 1])

    # 3. Train using Adam Optimizer
    costs = nn.train(train_X, train_Y, optimizer="adam", learning_rate=0.0007, epochs=5000)

    # 4. Evaluate Accuracy
    predictions = nn.predict(train_X)
    accuracy = np.mean(predictions == train_Y) * 100
    print(f"\nFinal Model Accuracy: {accuracy:.2f}%")

Cost after epoch 0: 0.684965
Cost after epoch 1000: 0.181558
Cost after epoch 2000: 0.169563
Cost after epoch 3000: 0.218465
Cost after epoch 4000: 0.154296

Final Model Accuracy: 94.33%
